In [39]:
"""
TransitCheck — feature extraction pipeline
==========================================

Turns a KOI catalogue row into six engineered features describing the shape of
its transit, for a gradient-boosted tree classifier.

Pipeline order per star:
    download -> clean -> flatten (transit-masked) -> fold -> bin -> measure

Feature summary:
    depth            how much the star dims during transit
    width            how long the dip lasts, measured at half-depth
    asymmetry        is the dip lopsided (ingress vs egress)
    snr              depth relative to the out-of-transit noise
    odd_even_diff    do odd- and even-numbered transits differ in depth
    secondary_depth  deepest dip anywhere outside the primary transit
"""

import numpy as np
import pandas as pd
import lightkurve as lk
import astropy.units as u


# ---------------------------------------------------------------------------
# Tunable constants
#
# These were chosen deliberately; see the notes beside each one before changing.
# ---------------------------------------------------------------------------

LOCAL_N_BINS = 201          # int   — bins across the local view (Shallue & Vanderburg convention)
LOCAL_WINDOW_DURATIONS = 4  # int   — local view spans this many transit durations, centred on the dip
DEPTH_WINDOW_FRACTION = 4   # int   — depth measured within duration/4 of centre (flat bottom only)
SHAPE_WINDOW_FRACTION = 2   # int   — width/asymmetry/snr use duration/2 (includes ingress & egress)
ODD_EVEN_N_BINS = 25        # int   — coarse on purpose: halved data, and only one number is needed
SECONDARY_BINS_PER_DURATION = 4    # int — bin width for the global view = duration / this
SECONDARY_BLANK_DURATIONS = 2      # int — blank out +/- this many durations around the primary
SECONDARY_THRESHOLD = 3            # int — a dip must sit this many noise-widths below baseline

OUTLIER_SIGMA_LOWER = 20    # int — lenient downward: protects real deep transits from being clipped
OUTLIER_SIGMA_UPPER = 5     # int — strict upward: removes cosmic rays and instrument glitches

FLATTEN_WINDOW_DURATIONS = 3  # int — detrending window spans this many transit durations
MIN_FLATTEN_WINDOW = 5        # int — Savitzky-Golay needs a window wider than its polynomial order

MAD_TO_SIGMA = 1.4826       # float — rescales a median-absolute-deviation onto a std-dev scale


# ---------------------------------------------------------------------------
# Generic helpers
# ---------------------------------------------------------------------------

def find_longest_true(mask):
    """Find the longest unbroken run of True values in a boolean array.

    A real dip is contiguous; a noise spike is a single isolated bin. Taking the
    longest run therefore ignores strays without needing to restrict where we look.

    In:  mask  — ndarray of bool
    Out: (start, end) as ints, where start is inclusive and end is exclusive,
         so mask[start:end] is exactly the run. Returns (None, None) if no True.
    """
    # Padding with False at both ends guarantees every run has an opening +1 and a
    # closing -1. The leading pad shifts indices right by one, np.diff shifts them
    # back left by one, so the returned indices land correctly on the original mask.
    padded = np.concatenate(([False], mask, [False]))   # ndarray of bool, len(mask) + 2
    changes = np.diff(padded.astype(int))               # ndarray of int,  len(mask) + 1

    starts = np.where(changes == 1)[0]    # ndarray of int — index where each run begins
    ends = np.where(changes == -1)[0]     # ndarray of int — index just past where each run ends

    if len(starts) == 0:
        return None, None

    lengths = ends - starts               # ndarray of int — length of each run
    longest = int(np.argmax(lengths))     # int — position of the winning run
    return int(starts[longest]), int(ends[longest])


def robust_noise(values):
    """Estimate the spread of an array, ignoring outliers.

    Uses the median absolute deviation rather than a standard deviation: one bad
    value, squared, dominates a std-dev, whereas a MAD barely moves. Scaling by
    1.4826 makes the result directly comparable to a std-dev for clean data.

    In:  values — ndarray of float (may contain NaN)
    Out: float. Returns np.nan if the spread can't be measured.
    """
    centre = np.nanmedian(values)                              # float
    spread = np.nanmedian(np.abs(values - centre))             # float
    noise = spread * MAD_TO_SIGMA                              # float

    if noise == 0 or np.isnan(noise):
        return np.nan
    return noise


# ---------------------------------------------------------------------------
# Stage 1 — download and clean
# ---------------------------------------------------------------------------

def load_star(kepid):
    """Download every available Kepler quarter for one star and clean it.

    In:  kepid — int, the Kepler Input Catalog ID
    Out: LightCurve, stitched across all quarters with outliers removed
    """
    search_result = lk.search_lightcurve(f"KIC {int(kepid)}", author='Kepler')
    star_lc = search_result.download_all().stitch()             # LightCurve

    # Asymmetric clipping: aggressive on upward spikes, forgiving downward so a
    # genuinely deep eclipse isn't mistaken for a glitch and thrown away.
    star_lc = star_lc.remove_outliers(
        sigma_lower=OUTLIER_SIGMA_LOWER,
        sigma_upper=OUTLIER_SIGMA_UPPER,
    )
    return star_lc


# ---------------------------------------------------------------------------
# Stage 2 — flatten, with the transit masked out of the trend fit
# ---------------------------------------------------------------------------

def build_transit_mask(star_lc, period, epoch_time, duration):
    """Flag which raw data points fall inside a transit.

    Note this predicts transit times arithmetically from the catalogue's period
    and epoch — it never inspects the flux. Folding is used only as a convenient
    way to compute "distance from the nearest transit centre" for every point at
    once, and time_original translates the selection back to real timestamps.

    In:  star_lc     — LightCurve (unfolded)
         period      — float, days
         epoch_time  — float, BKJD
         duration    — float, hours
    Out: ndarray of bool, aligned index-for-index with star_lc
    """
    # normalize_phase=True makes .phase a fraction of the period (-0.5 to 0.5)
    # rather than days, so it can be compared against fractional_duration directly.
    temp_fold = star_lc.fold(period=period, epoch_time=epoch_time, normalize_phase=True)

    fractional_duration = (duration / 24) / period              # float — transit length as a fraction of one orbit
    phase_mask = np.abs(temp_fold.phase.value) < (fractional_duration * 1.5)   # ndarray of bool, in folded order

    # np.isin is a membership test, so it doesn't care that folding reordered the
    # rows — the result comes back in star_lc's own order and length.
    return np.isin(star_lc.time.value, temp_fold.time_original.value[phase_mask])


def flatten_star(star_lc, transit_mask, duration):
    """Remove slow brightness drift without eating into the transit itself.

    The detrending window is sized in data points, so it has to be derived from
    this star's own cadence: too narrow and the fit follows the dip, too wide and
    it can't track the star's real variability.

    In:  star_lc      — LightCurve (unfolded)
         transit_mask — ndarray of bool
         duration     — float, hours
    Out: LightCurve, flattened
    """
    cadence_minutes = np.median(np.diff(star_lc.time.value)) * 24 * 60   # float — gap between consecutive points

    window_length = int((FLATTEN_WINDOW_DURATIONS * 60 * duration) / cadence_minutes)   # int — window in data points
    window_length = max(window_length, MIN_FLATTEN_WINDOW)
    window_length += 1 if window_length % 2 == 0 else 0    # Savitzky-Golay windows must be odd

    # mask= tells the trend fit to ignore in-transit points, so the fit has no
    # reason to dip toward the transit (or overshoot recovering from it).
    return star_lc.flatten(window_length=window_length, mask=transit_mask)


# ---------------------------------------------------------------------------
# Stage 3 — fold, slice to the transit window, bin
# ---------------------------------------------------------------------------

def prepare_transit(lc, period, epoch_time, duration,
                    n_bins=LOCAL_N_BINS,
                    window_durations=LOCAL_WINDOW_DURATIONS):
    """Fold, cut out the region around the transit, bin it, and measure depth.

    Everything downstream works from these outputs rather than recomputing the
    fold and bin, so all the shape features describe the same curve.

    In:  lc               — LightCurve (flattened, unfolded)
         period           — float, days
         epoch_time       — float, BKJD
         duration         — float, hours
         n_bins           — int, bins across the whole window
         window_durations — int, window width in transit durations
    Out: (depth, flux, phase, baseline)
         depth    — float, fractional dip below baseline
         flux     — ndarray of float, one value per bin
         phase    — ndarray of float, days from transit centre, one per bin
         baseline — float, the out-of-transit flux level
    """
    folded = lc.fold(period=period, epoch_time=epoch_time)      # FoldedLightCurve, phase in days

    half_window_days = (duration * window_durations / 2) / 24   # float — half the window, hours -> days
    window_mask = np.abs(folded.phase.value) < half_window_days  # ndarray of bool
    local_view = folded[window_mask]                             # FoldedLightCurve, just the transit region

    bin_width = (duration * window_durations / n_bins) * u.hour  # Quantity — window divided into n_bins pieces
    binned = local_view.bin(time_bin_size=bin_width)             # FoldedLightCurve, one row per bin

    flux = binned.flux.value      # ndarray of float
    phase = binned.phase.value    # ndarray of float, days

    # duration/4 keeps to the flat bottom of the dip. Wider drags the median up
    # toward baseline (the sloped edges) and undermeasures the depth.
    in_transit = np.abs(phase) < (duration / 24) / DEPTH_WINDOW_FRACTION   # ndarray of bool

    baseline = np.nanmedian(flux[~in_transit])                   # float
    depth = baseline - np.nanmedian(flux[in_transit])            # float

    return depth, flux, phase, baseline


# ---------------------------------------------------------------------------
# Stage 4 — the six features
# ---------------------------------------------------------------------------

def measure_width(flux, phase, baseline, depth):
    """How long the dip lasts, measured where it crosses half its depth.

    Uses the longest contiguous run rather than counting all below-threshold bins
    (a stray inflates the count) or spanning first-to-last (a stray at the edge
    stretches it enormously).

    In:  flux, phase — ndarrays of float from prepare_transit
         baseline, depth — floats from prepare_transit
    Out: float, hours. np.nan if nothing crosses the threshold.
    """
    below_half_depth = flux < (baseline - depth / 2)             # ndarray of bool
    start, end = find_longest_true(below_half_depth)

    if start is None:
        return np.nan

    # end is exclusive, so the last bin in the run is at end - 1.
    return (phase[end - 1] - phase[start]) * 24                  # float, days -> hours


def measure_asymmetry(flux, phase, baseline, duration):
    """Is the dip deeper on the way in than on the way out?

    A real planet transit is symmetric, since the geometry going in mirrors the
    geometry coming out. Signed rather than absolute: a tree can ignore the sign
    by splitting at zero, but can't recover it if it's been thrown away.

    In:  flux, phase — ndarrays of float from prepare_transit
         baseline    — float from prepare_transit
         duration    — float, hours
    Out: float. Positive means ingress (left) is deeper than egress (right).
    """
    # duration/2 here, not /4 — asymmetry lives in the sloped edges, which the
    # narrower depth window deliberately excludes.
    in_transit = np.abs(phase) < (duration / 24) / SHAPE_WINDOW_FRACTION   # ndarray of bool

    ingress = in_transit & (phase < 0)                           # ndarray of bool
    egress = in_transit & (phase > 0)                            # ndarray of bool

    ingress_depth = baseline - np.nanmedian(flux[ingress])       # float
    egress_depth = baseline - np.nanmedian(flux[egress])         # float

    return ingress_depth - egress_depth


def measure_snr(flux, phase, depth, duration):
    """Depth relative to how much the flux wobbles when nothing is happening.

    A 200 ppm dip on a quiet star is convincing; the same dip on a noisy star is
    meaningless. Depth alone can't tell those apart.

    In:  flux, phase — ndarrays of float from prepare_transit
         depth       — float from prepare_transit
         duration    — float, hours
    Out: float. np.nan if the noise level can't be measured.
    """
    # duration/2 so the "noise" sample sits clear of the ingress/egress slopes,
    # which are real signal and would inflate the estimate.
    out_of_transit = np.abs(phase) > (duration / 24) / SHAPE_WINDOW_FRACTION   # ndarray of bool

    noise = robust_noise(flux[out_of_transit])                   # float

    if np.isnan(noise):
        return np.nan
    return depth / noise


def measure_odd_even_diff(flat_star_lc, transit_mask, period, epoch_time, duration,
                          n_bins=ODD_EVEN_N_BINS):
    """Compare the depth of odd-numbered transits against even-numbered ones.

    A real planet's transits are identical whichever one you look at. A blended
    eclipsing binary often alternates between a deeper and shallower dip.

    Splitting has to happen before folding: folding stacks every transit onto one
    shared phase axis, which erases which transit a point came from.

    In:  flat_star_lc — LightCurve (flattened, unfolded)
         transit_mask — ndarray of bool, aligned with flat_star_lc
         period, epoch_time, duration — floats
         n_bins       — int, coarse since each group holds only half the data
    Out: float, absolute difference between the two depths
    """
    # Dividing elapsed time by the period gives two things at once: the whole
    # number is which transit, the fractional leftover is the phase. Here we want
    # the whole number.
    transit_number = np.round(
        (flat_star_lc.time.value - epoch_time) / period
    ).astype(int)                                                # ndarray of int, one per data point

    # Both conditions must hold: in-transit at all, and the right parity.
    even_filter = transit_mask & (transit_number % 2 == 0)       # ndarray of bool
    odd_filter = transit_mask & (transit_number % 2 == 1)        # ndarray of bool

    even_depth, _, _, _ = prepare_transit(
        flat_star_lc[even_filter], period, epoch_time, duration, n_bins=n_bins
    )
    odd_depth, _, _, _ = prepare_transit(
        flat_star_lc[odd_filter], period, epoch_time, duration, n_bins=n_bins
    )

    return abs(even_depth - odd_depth)


def measure_secondary_depth(lc, period, epoch_time, duration,
                            bins_per_duration=SECONDARY_BINS_PER_DURATION,
                            threshold=SECONDARY_THRESHOLD):
    """Find the deepest dip anywhere in the orbit other than the primary transit.

    An eclipsing binary eclipses twice per orbit; a planet effectively once. A
    detectable secondary is therefore strong evidence of two stars.

    This needs the whole folded period, not the local view — a secondary sits
    roughly half a period away, far outside the local window. Eccentric orbits
    shift it off-centre, so we search everywhere rather than checking phase 0.5.

    In:  lc — LightCurve (flattened, unfolded)
         period, epoch_time, duration — floats
         bins_per_duration — int, sets bin width so a secondary spans several bins
         threshold         — int, how many noise-widths below baseline counts as a dip
    Out: float, fractional depth. 0.0 when no secondary is found.
    """
    folded = lc.fold(period=period, epoch_time=epoch_time)       # FoldedLightCurve

    # Bin width comes from the physics, not a fixed count: the array collapses to
    # one number, so equal-length output across stars doesn't matter here.
    bin_width = (duration / bins_per_duration) * u.hour          # Quantity
    binned = folded.bin(time_bin_size=bin_width)                 # FoldedLightCurve

    flux = binned.flux.value      # ndarray of float
    phase = binned.phase.value    # ndarray of float, days

    # Blank generously: the primary's sloped edges are deep enough to win any
    # "find the deepest dip" search and be misreported as a secondary.
    outside_primary = np.abs(phase) > (duration / 24) * SECONDARY_BLANK_DURATIONS   # ndarray of bool

    baseline = np.nanmedian(flux[outside_primary])               # float
    noise = robust_noise(flux[outside_primary])                  # float

    if np.isnan(noise):
        return 0.0

    # The & with outside_primary is what stops the run-finder latching onto the
    # primary transit, which is far below any threshold we'd set.
    below_threshold = outside_primary & (flux < (baseline - threshold * noise))   # ndarray of bool
    start, end = find_longest_true(below_threshold)

    if start is None:
        return 0.0    # a real answer ("no secondary"), not a failure

    return baseline - np.nanmedian(flux[start:end])


# ---------------------------------------------------------------------------
# Stage 5 — one star, all six features
# ---------------------------------------------------------------------------

def extract_features(kepid, period, epoch_time, duration):
    """Run the whole pipeline for one KOI and return its six features.

    In:  kepid       — int
         period      — float, days      (koi_period)
         epoch_time  — float, BKJD      (koi_time0bk)
         duration    — float, hours     (koi_duration)
    Out: dict of feature name -> float
    """
    star_lc = load_star(kepid)

    transit_mask = build_transit_mask(star_lc, period, epoch_time, duration)   # ndarray of bool
    flat_star_lc = flatten_star(star_lc, transit_mask, duration)               # LightCurve

    depth, flux, phase, baseline = prepare_transit(flat_star_lc, period, epoch_time, duration)

    return {
        'depth': depth,
        'width': measure_width(flux, phase, baseline, depth),
        'asymmetry': measure_asymmetry(flux, phase, baseline, duration),
        'snr': measure_snr(flux, phase, depth, duration),
        'odd_even_diff': measure_odd_even_diff(
            flat_star_lc, transit_mask, period, epoch_time, duration
        ),
        'secondary_depth': measure_secondary_depth(
            flat_star_lc, period, epoch_time, duration
        ),
    }


# ---------------------------------------------------------------------------
# Stage 6 — run across a batch
# ---------------------------------------------------------------------------

def build_feature_table(koi_batch):
    """Extract features for every row of a KOI table.

    In:  koi_batch — DataFrame with a clean 0..n-1 index and the KOI columns
    Out: DataFrame, one row per star
    """
    rows = []    # list of dict

    for i in range(len(koi_batch)):
        features = extract_features(
            kepid=koi_batch.loc[i, 'kepid'],
            period=koi_batch.loc[i, 'koi_period'],
            epoch_time=koi_batch.loc[i, 'koi_time0bk'],
            duration=koi_batch.loc[i, 'koi_duration'],
        )

        # kepid and disposition are carried through for inspection and for
        # grouping the train/test split by star later. They are not features.
        # koi_depth is kept only as a ground-truth check on the depth measurement
        # and must be dropped before training — it comes from the same vetting
        # process that produced the labels.
        rows.append({
            'kepid': koi_batch.loc[i, 'kepid'],
            'disposition': koi_batch.loc[i, 'koi_disposition'],
            'koi_depth_ppm': koi_batch.loc[i, 'koi_depth'],
            'koi_duration_hours': koi_batch.loc[i, 'koi_duration'],
            **features,
        })

    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# Optional — plot one star's local view, for eyeballing
# ---------------------------------------------------------------------------

def plot_local_view(kepid, period, epoch_time, duration, label=None):
    """Download, clean, flatten, fold and bin one star, then plot it.

    Fixed bin count is correct here (unlike in the feature functions): consistent
    resolution across stars is what makes plots comparable by eye.

    In:  kepid, period, epoch_time, duration — as above
         label — str or None, legend text
    Out: matplotlib Axes
    """
    star_lc = load_star(kepid)
    transit_mask = build_transit_mask(star_lc, period, epoch_time, duration)
    flat_star_lc = flatten_star(star_lc, transit_mask, duration)

    folded = flat_star_lc.fold(period=period, epoch_time=epoch_time)

    half_window_days = (duration * LOCAL_WINDOW_DURATIONS / 2) / 24
    local_view = folded[np.abs(folded.phase.value) < half_window_days]

    bin_width = (duration * LOCAL_WINDOW_DURATIONS / LOCAL_N_BINS) * u.hour
    binned = local_view.bin(time_bin_size=bin_width)

    return binned.plot(label=label or f"KIC {int(kepid)}")


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

if __name__ == '__main__':
    koi_table = pd.read_csv("MyProject_sync.csv")               # DataFrame

    confirmed = koi_table[koi_table['koi_disposition'] == 'CONFIRMED'].head(5)
    false_positive = koi_table[koi_table['koi_disposition'] == 'FALSE POSITIVE'].head(5)

    # ignore_index=True gives a clean 0..9 index so .loc[i] means the i-th row.
    test_batch = pd.concat([confirmed, false_positive], axis=0, ignore_index=True)

    feature_table = build_feature_table(test_batch)
    print(feature_table)

      kepid     disposition  koi_depth_ppm  koi_duration_hours  \
0  10797460       CONFIRMED          615.8             2.95750   
1  10797460       CONFIRMED          874.8             4.50700   
2  10854555       CONFIRMED          603.3             1.65450   
3  10872983       CONFIRMED         1517.5             4.59450   
4  10872983       CONFIRMED          686.0             3.14020   
5  10848459  FALSE POSITIVE         8079.2             2.40641   
6   6721123  FALSE POSITIVE          233.7             5.02200   
7  10419211  FALSE POSITIVE        17984.3             3.63990   
8  10464078  FALSE POSITIVE         8918.7            12.21550   
9  10480982  FALSE POSITIVE        74284.0             4.79843   

                    depth     width                asymmetry  \
0   0.0005832201699911277  2.413085   3.6485220229587334e-05   
1     0.00074731919222637  2.780438  -0.00011057669369962664   
2     0.00051689670707189  0.493881   -7.115462249240867e-05   
3    0.0014782547

In [1]:
%matplotlib
from astropy import units as u
from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive
import lightkurve as lk
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import time

Using matplotlib backend: module://matplotlib_inline.backend_inline


c:\Users\Guest1\githubfiles\TransitCheck-Project\.venv\Lib\site-packages\lightkurve\prf\__init__.py:7: UserWarning: Warning: the tpfmodel submodule is not available without oktopus installed, which requires a current version of autograd. See #1452 for details.
  warnings.warn(


In [ ]:
sample_index = np.where(my_lcList.mission == 'Kepler Quarter 13')
sample = my_lcList[sample_index]
sample

In [ ]:
abc = lk.search_lightcurve('Kepler 10', author='Kepler', quarter=13, exptime='short')
abc.download_all().plot()



In [ ]:
my_tpf = lk.search_targetpixelfile('K2-199', exptime='long')
adef = my_tpf[0].download()
adef.plot()

In [22]:
dfTable2 = pd.read_csv("MyProject_sync.csv") #used to be mainTrainTable2. We use this.
# dfTable2

In [23]:
dft_slice1 = (dfTable2['koi_disposition'] == 'CONFIRMED')
dft_slice2 = (dfTable2['koi_disposition'] == 'FALSE POSITIVE')
confirmed_batch = dfTable2[dft_slice1].head(5)
fp_batch = dfTable2[dft_slice2].head(5)
dft_test_batch = pd.concat([confirmed_batch, fp_batch], axis=0, ignore_index=True)
dft_test_batch

,kepid,kepoi_name,kepler_name,ra,ra_err,ra_str,dec,dec_err,dec_str,koi_gmag,...,koi_fpflag_co,koi_fpflag_ec,koi_insol,koi_insol_err1,koi_insol_err2,koi_srho,koi_srho_err1,koi_srho_err2,koi_fittype,koi_score
0,10797460,K00752.01,Kepler-227 b,291.93423,0.0,19h27m44.22s,48.141651,0.0,+48d08m29.9s,15.890,...,0,0,93.59,29.45,-16.65,3.20796,0.33173,-1.09986,LS+MCMC,1.000
1,10797460,K00752.02,Kepler-227 c,291.93423,0.0,19h27m44.22s,48.141651,0.0,+48d08m29.9s,15.890,...,0,0,9.11,2.87,-1.62,3.02368,2.20489,-2.49638,LS+MCMC,0.969
2,10854555,K00755.01,Kepler-664 b,288.75488,0.0,19h15m01.17s,48.226200,0.0,+48d13m34.3s,16.015,...,0,0,926.16,874.33,-314.24,1.98635,2.71141,-1.74541,LS+MCMC,1.000
3,10872983,K00756.01,Kepler-228 d,296.28613,0.0,19h45m08.67s,48.224670,0.0,+48d13m28.8s,16.234,...,0,0,114.81,112.85,-36.70,0.67324,0.33286,-0.38858,LS+MCMC,1.000
4,10872983,K00756.02,Kepler-228 c,296.28613,0.0,19h45m08.67s,48.224670,0.0,+48d13m28.8s,16.234,...,0,0,427.65,420.33,-136.70,0.37377,0.74768,-0.26357,LS+MCMC,1.000
5,10848459,K00754.01,NaN,285.53461,0.0,19h02m08.31s,48.285210,0.0,+48d17m06.8s,16.100,...,0,0,891.96,668.95,-230.35,0.22080,0.00917,-0.01837,LS+MCMC,0.000
6,6721123,K00114.01,NaN,298.86435,0.0,19h55m27.44s,42.151569,0.0,+42d09m05.6s,12.999,...,1,0,767.22,349.28,-365.49,0.00485,0.27864,-0.00053,LS+MCMC,0.000
7,10419211,K00742.01,NaN,297.07993,0.0,19h48m19.18s,47.597401,0.0,+47d35m50.6s,15.994,...,0,0,75.88,58.89,-19.99,1.27376,2.30242,-0.41065,LS+MCMC,0.000
8,10464078,K00743.01,NaN,289.25821,0.0,19h17m01.97s,47.635319,0.0,+47d38m07.1s,16.229,...,0,0,17.69,6.66,-4.88,0.03785,0.00107,-0.00172,LS+MCMC,0.000
9,10480982,K00744.01,NaN,295.81454,0.0,19h43m15.49s,47.690350,0.0,+47d41m25.3s,15.798,...,0,0,55.97,54.55,-16.40,1.57359,0.00657,-0.02917,LS+MCMC,0.000


In [25]:
for i in range(len(dft_test_batch)):
    # search up a star's lightvurve data, download it, stitch it. and remove the very high and low points
    group_lc = lk.search_lightcurve(f"KIC {int(dft_test_batch.loc[i, 'kepid'])}", author='Kepler')
    star_lc = group_lc.download_all().stitch()
    star_lc = star_lc.remove_outliers(sigma_lower=20, sigma_upper=5)

    # gathering my parameters for folding and flattening
    period = dft_test_batch.loc[i, 'koi_period'] # transit time as fraction of orbital time: this is in days
    epoch_time = dft_test_batch.loc[i, 'koi_time0bk'] # time0 of the whole thing, the beginning: this is in BKJD
    duration = dft_test_batch.loc[i, 'koi_duration'] # transit duration: this is in hours
    cadence = np.median(np.diff(star_lc.time.value)) * 24 * 60 # time gap between points, or the kind of exposure used (whether long or short) in minutes
    # bin_duration = duration * 0.075
    # if (bin_duration * 60) < cadence:
    #     bin_duration = cadence / 60  # back to hours
    window_length = int((3 * 60 * duration) / cadence) # for flattening
    window_length = max(window_length, 5)
    window_length += 1 if window_length % 2 == 0 else 0

    # using the parameters to construct a mask for the transit for flattening 
    temp_fold = star_lc.fold(period=period, epoch_time=epoch_time, normalize_phase=True)
    fractional_duration = (duration / 24) / period
    phase_mask = np.abs(temp_fold.phase.value) < (fractional_duration * 1.5)
    transit_mask = np.isin(star_lc.time.value, temp_fold.time_original.value[phase_mask])

    # actual folding and flattening
    flat_star_lc = star_lc.flatten(window_length=window_length, mask=transit_mask)
    folded_star_lc = flat_star_lc.fold(period=period, epoch_time=epoch_time)

    # preparing parameters and mask for binning
    n_bins = 201            # from Shallue & Vanderburg's local view
    bin_window_durations = 4    # how many durations for our zoomed-in slice
    bin_half_window = (duration * bin_window_durations / 2) / 24  # in days
    lcl_mask = np.abs(folded_star_lc.phase.value) < bin_half_window
    lcl_view = folded_star_lc[lcl_mask]

    # calculating bin width
    bin_width = (duration * bin_window_durations / n_bins) * u.hour

    # actual binning and plotting
    # binned_star_lc = folded_star_lc.bin(time_bin_size=bin_duration * u.hour)
    lcl_binned_star_lc = lcl_view.bin(time_bin_size=bin_width)
    # ax = lcl_binned_star_lc.plot(label=f'Binned Light Curve {i}')


In [37]:

# To compute depth, width and asymmetry
def measure_depth(lc, period, epoch_time, duration, n_bins, window_durations=4):
    """Fold, slice to the transit window, bin, and return the dip depth."""
    folded = lc.fold(period=period, epoch_time=epoch_time)

    half_window_days = (duration * window_durations / 2) / 24
    mask = np.abs(folded.phase.value) < half_window_days
    local = folded[mask]

    bin_width = (duration * window_durations / n_bins) * u.hour
    binned = local.bin(time_bin_size=bin_width)

    flux = binned.flux.value
    phase = binned.phase.value

    in_transit = np.abs(phase) < (duration / 24) / 4
    baseline = np.nanmedian(flux[~in_transit])
    dip = np.nanmedian(flux[in_transit])
    return baseline - dip

transit_number = np.round((flat_star_lc.time.value - epoch_time) / period).astype(int)
even_filter = transit_mask & ((transit_number % 2) == 0)   
odd_filter  = transit_mask & ((transit_number % 2) == 1)   

even_lc = flat_star_lc[even_filter]  
odd_lc  = flat_star_lc[odd_filter]

oe_bins = 25   # coarse on purpose — one number needed, not a resolved shape

even_depth = measure_depth(even_lc, period, epoch_time, duration, oe_bins)
odd_depth  = measure_depth(odd_lc,  period, epoch_time, duration, oe_bins)

odd_even_diff = abs(even_depth - odd_depth)

print(dft_test_batch.loc[i, ['kepid', 'koi_disposition', 'koi_depth']])
print(even_depth, odd_depth, odd_even_diff)


# print(transit_number[500:1500])         # should be whole numbers, mostly repeating
# print(len(transit_number) == len(transit_mask))   # must be True for & to work

kepid                    10480982
koi_disposition    FALSE POSITIVE
koi_depth                 74284.0
Name: 9, dtype: object
0.06325709789706557 0.06295505176836991 0.0003020461286956655


In [38]:
results = []

for i in range(len(dft_test_batch)):
    group_lc = lk.search_lightcurve(f"KIC {int(dft_test_batch.loc[i, 'kepid'])}", author='Kepler')
    star_lc = group_lc.download_all().stitch()
    star_lc = star_lc.remove_outliers(sigma_lower=20, sigma_upper=5)

    period = dft_test_batch.loc[i, 'koi_period']
    epoch_time = dft_test_batch.loc[i, 'koi_time0bk']
    duration = dft_test_batch.loc[i, 'koi_duration']

    cadence = np.median(np.diff(star_lc.time.value)) * 24 * 60
    window_length = int((3 * 60 * duration) / cadence)
    window_length = max(window_length, 5)
    window_length += 1 if window_length % 2 == 0 else 0

    temp_fold = star_lc.fold(period=period, epoch_time=epoch_time, normalize_phase=True)
    fractional_duration = (duration / 24) / period
    phase_mask = np.abs(temp_fold.phase.value) < (fractional_duration * 1.5)
    transit_mask = np.isin(star_lc.time.value, temp_fold.time_original.value[phase_mask])

    flat_star_lc = star_lc.flatten(window_length=window_length, mask=transit_mask)

    transit_number = np.round((flat_star_lc.time.value - epoch_time) / period)
    even_filter = transit_mask & ((transit_number % 2) == 0)
    odd_filter  = transit_mask & ((transit_number % 2) == 1)

    even_depth = measure_depth(flat_star_lc[even_filter], period, epoch_time, duration, 25)
    odd_depth  = measure_depth(flat_star_lc[odd_filter],  period, epoch_time, duration, 25)
    full_depth = measure_depth(flat_star_lc, period, epoch_time, duration, 201)

    results.append({
        'kepid': dft_test_batch.loc[i, 'kepid'],
        'disposition': dft_test_batch.loc[i, 'koi_disposition'],
        'koi_depth_ppm': dft_test_batch.loc[i, 'koi_depth'],
        'measured_depth': full_depth,
        'odd_even_diff': abs(even_depth - odd_depth),
    })

pd.DataFrame(results)

,kepid,disposition,koi_depth_ppm,measured_depth,odd_even_diff
0,10797460,CONFIRMED,615.8,0.0005832201699911277,4.473484742428813e-05
1,10797460,CONFIRMED,874.8,0.00074731919222637,0.0002477590467693247
2,10854555,CONFIRMED,603.3,0.00051689670707189,3.651618539368684e-05
3,10872983,CONFIRMED,1517.5,0.001478254711603011,0.0002782854695883019
4,10872983,CONFIRMED,686.0,0.0006345418126378899,3.5945774938594965e-05
5,10848459,FALSE POSITIVE,8079.2,0.007046340695659681,0.00048199094838441425
6,6721123,FALSE POSITIVE,233.7,0.00017954142046872068,3.201490837334564e-05
7,10419211,FALSE POSITIVE,17984.3,0.01596446748595548,2.8785611772996766e-05
8,10464078,FALSE POSITIVE,8918.7,0.00970401614484806,0.00027689289727772515
9,10480982,FALSE POSITIVE,74284.0,0.06632597153183628,0.0003020461286956655


In [ ]:
# for i in range(len(dft_test_batch)):
#     group_lc = lk.search_lightcurve(f"KIC {int(dft_test_batch.loc[i, 'kepid'])}", author='Kepler')
#     star_lc = group_lc.download_all().stitch()
#     # star_lc = star_lc.remove_outliers(sigma_lower=20, sigma_upper=5)
#     print(f'{dft_test_batch.loc[i, 'kepid']}, {dft_test_batch.loc[i, 'koi_disposition']}, {len(star_lc)}')

d = np.array([0, 1, 0, 0, -1, 0])
# np.where(d == 1)     →  (array([1]),)
# np.where(d == -1)    →  (array([4]),)

def find_longest_True(mask: Any) -> ndarray[_AnyShape, _dtype]:
    padded = np.concatenate(([False], mask, [False]))
    d = np.diff(padded.astype(int))
    starts = np.where(d == 1)[0]
    ends   = np.where(d == -1)[0]
    index = 0
    length = 0
    for i in len(starts):
        if ends[i] - starts[i] > length:
            index = i
            length = ends[i] - starts[i]



#     return d[starts[index]:ends[index + 1]]

In [ ]:
def measure_width(lc, period, epoch_time, duration, n_bins, window_durations=4):
    """Fold, slice, bin, then measure how wide the dip is at half-depth."""
    
    # --- same setup as measure_depth: fold, slice to window, bin ---
    # (you have these four lines already)
    
    flux = binned.flux.value
    phase = binned.phase.value
    
    # --- get baseline and depth (same in_transit logic as measure_depth) ---
    
    # --- build the boolean array: which bins sit below half-depth? ---
    below = ...
    
    # --- find the longest unbroken run ---
    start, end = find_longest_true(below)
    
    # --- handle the empty case ---
    if start is None:
        return np.nan
    
    # --- convert indices to a duration ---
    # remember: end is exclusive, phase is in days, koi_duration is in hours
    width_hours = ...
    
    return width_hours


def measure_width(flux, phase, baseline, depth):
    below = flux < (baseline - depth / 2)
    start, end = find_longest_true(below)
    if start is None:
        return np.nan
    return (phase[end - 1] - phase[start]) * 24


def measure_snr(flux, phase, baseline, depth, duration):
    out_of_transit = np.abs(phase) > (duration / 24) / 2
    values = flux[out_of_transit]
    med = np.nanmedian(values)
    noise = np.nanmedian(np.abs(values - med)) * 1.4826
    if noise == 0 or np.isnan(noise):
        return np.nan
    return depth / noise

In [32]:
# def isPrime(num: int) -> bool:
#     if num < 2:
#         return False
#     if num in (2, 3):
#         return True
#     if num % 2 == 0 or num % 3 == 0:
#         return False
#     i = 5
#     while i * i <= num:
#         if num % i == 0 or num % (i + 2) == 0:
#             return False
#         i += 6
#     return True

# toRun = pow(2, 136279841) - 1

# print(isPrime(7547))
# print(isPrime(1223))
# print(isPrime(7))
# print(isPrime(9974))
# print(isPrime(toRun))
